<a href="https://colab.research.google.com/github/rubenoliva-dominguez/curso_cartagena/blob/main/dia4-1/5_decoder_sft_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset

In [5]:
from datasets import load_dataset, DatasetDict

# Cargamos el dataset CoNaLa
# Conala es un dataset de pares (descripción en inglés, código en Python)
# lo usaremos para hacer SFT
train = load_dataset("json", data_files="conala-paired-train.json")['train']
test = load_dataset("json", data_files="conala-paired-test.json")['train']
dataset = DatasetDict({
    "train": train,
    "test": test,
})

In [6]:
# prompt a usar
PROMPT = """### Instruction:
{instruction}

### Response:
{response}"""

In [7]:
# Filtramos los ejemplos que no tienen reescritura de la intención
dataset = dataset.filter(lambda x: x["rewritten_intent"] != None)
dataset

Filter:   0%|          | 0/2379 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['intent', 'rewritten_intent', 'snippet', 'question_id'],
        num_rows: 2300
    })
    test: Dataset({
        features: ['intent', 'rewritten_intent', 'snippet', 'question_id'],
        num_rows: 477
    })
})

In [8]:
# SFT requiere que el dataset tenga las columnas 'prompt' y 'completion'
# El prompt será la intención reescrita y el completion el snippet de código a predecir
# SFT entrena el modelo para predecir la completion dado el prompt
def preprocess_function(example):
    return {
        "prompt": PROMPT.format(instruction=example['rewritten_intent'], response=""),
        "completion": example['snippet']
    }

dataset = dataset.map(preprocess_function, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/2300 [00:00<?, ? examples/s]

Map:   0%|          | 0/477 [00:00<?, ? examples/s]

In [9]:
print(dataset["test"]["prompt"][0])
print("------------------")
print(dataset["test"]["completion"][0])

### Instruction:
send a signal `signal.SIGUSR1` to the current process

### Response:

------------------
os.kill(os.getpid(), signal.SIGUSR1)


# Entrenamiento

In [12]:
!pip install -U trl
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
import torch

# Configuramos el entrenamiento
training_args = SFTConfig(
    output_dir="qwen-sft-lora",
    model_init_kwargs={"dtype": torch.bfloat16}, # usar bfloat16 para que el modelo ocupe menos memoria
    report_to="none",
    num_train_epochs=1, # lo dejamos a 1 para que sea rápido
    max_length=256, # es un dataset instrucción + una línea de código, no hace falta que sea muy largo
    per_device_train_batch_size=4, # batch size pequeño para que quepa en GPU
    gradient_accumulation_steps=8, # acumulamos gradientes para simular un batch size mayor
    eval_strategy="epoch", # estrategia de evaluación por pasos, pasos=veces que se modifican los pesos
    save_strategy="epoch", # estrategia de guardado por pasos
    save_total_limit=1, # guardamos solo el último checkpoint para no llenar el disco
    learning_rate=3e-4, # tasa de aprendizaje un poco más alta para LoRA
)

# instanciamos el trainer y entrenamos
trainer = SFTTrainer(
    model="Qwen/Qwen2.5-1.5B",
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    peft_config=LoraConfig() # usamos LoRA para hacer fine-tuning eficientemente con los parametros por defecto
)
trainer.train()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 15.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

Tokenizing eval dataset:   0%|          | 0/477 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/477 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,0.725644,0.745054


TrainOutput(global_step=72, training_loss=0.8070371382766299, metrics={'train_runtime': 544.0002, 'train_samples_per_second': 4.228, 'train_steps_per_second': 0.132, 'total_flos': 912818328870912.0, 'train_loss': 0.8070371382766299})

# Evaluación

In [17]:
from transformers import pipeline

# probamos el modelo fine-tuneado
pipe = pipeline("text-generation", model="qwen-sft-lora/checkpoint-72", temperature=0.1, do_sample=True)
instruction = "concatenate [Pedro, Es, guay]" #"compute the mean of the list [1, 2, 3, 4, 5]"
prompt = PROMPT.format(instruction=instruction, response="")
response = pipe(prompt)

print(response[0]["generated_text"])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
concatenate [Pedro, Es, guay]

### Response:
'Pedro Es guay'
